In [15]:
import requests
import json
import re

url = "https://www.cricbuzz.com/profiles/9311/jasprit-bumrah"
headers = {"User-Agent": "Mozilla/5.0"}

resp = requests.get(url, headers=headers)
print("Status:", resp.status_code)

# Find all JSON-LD blocks
json_blocks = re.findall(
    r'<script type="application/ld\+json">(.*?)</script>',
    resp.text,
    re.S
)

print("JSON-LD blocks found:", len(json_blocks))

for block in json_blocks:
    try:
        data = json.loads(block)

        # Look for Person entity
        if isinstance(data, dict) and "mainEntity" in data:
            person = data["mainEntity"]
            if person.get("type") == "Person":
                print("\n=== PLAYER DATA FOUND ===")
                for k, v in person.items():
                    print(f"{k}: {v}")

    except json.JSONDecodeError:
        continue


Status: 200
JSON-LD blocks found: 6

=== PLAYER DATA FOUND ===
type: Person
name: Jasprit Bumrah
description: In an Indian team desperately searching for a death overs' bowler, Jasprit Bumrah came to the fore through the Indian Premier League, as a boon for cricket in the country. The scantily-built pacer from Gujarat has managed to perfect the art of bowling inch-perfect yorkers as an understudy to Lasith Malinga as a part of the Mumbai Indians franchise and has grown into an indispensable asset for the Indian team in the limited-overs format.  <br/>  <br/> Having been a consistent performer in the domestic circuit, Bumrah has been a menace to the batsmen at the Ranji level since his debut. His quick-arm action has made his variations almost indiscernible - starting from his slower deliveries to his reverse-swinging yorkers at the death. An injury kept him out of action for a while in the 2014/15 seasons but he returned and continued to perform unaffected by the injury break. A nation

In [12]:
keywords = ["Born", "Bowling", "Role", "Batting", "Height"]

for kw in keywords:
    print(kw, "->", kw in resp.text)


Born -> True
Bowling -> True
Role -> True
Batting -> True
Height -> False


In [13]:
classes = set()

for div in soup.find_all("div"):
    cls = div.get("class")
    if cls:
        classes.update(cls)

print(sorted(classes))


['-translate-x-full', 'bg-[#0000001F]', 'bg-[#444]', 'bg-[#4a4a4a]', 'bg-[#E6F6EB]', 'bg-[#FAFAFA]', 'bg-cbGrnCyn', 'bg-cbGrpHdrBkg', 'bg-cbLightGrayish', 'bg-cbOffWhite', 'bg-cbWhite', 'bg-gray-200', 'bg-white', 'block', 'border', 'border-[#222]', 'border-b', 'border-cbBorderGrey', 'border-dashed', 'border-gray-200', 'border-gray-300', 'border-r', 'bottom-0', 'cb-side-menu-parent', 'cbCloseIcon', 'cbCrtIcnDark', 'cbLogoSvg', 'cbNoScrollbar', 'cbPlusIco', 'col-span-1', 'col-span-2', 'col-span-3', 'col-span-5', 'columns-4', 'cursor-pointer', 'dark:bg-cbBkgDark', 'dark:bg-cbGrpHdrBkgDark', 'dark:bg-cbHdrBkgDark', 'dark:bg-cbItmBkgDark', 'dark:cbCrtIcn', 'dark:text-cbTxtSec', 'false', 'fixed', 'flex', 'flex-1', 'flex-col', 'flex-grow', 'flex-shrink-0', 'font-bold', 'font-medium', 'font-semibold', 'gap-1', 'gap-2', 'gap-3', 'gap-4', 'gap-6', 'gap-px', 'gap-y-2', 'gap-y-px', 'grid', 'grid-cols-12', 'grid-cols-2', 'grid-cols-3', 'grid-cols-[1fr_0.6fr_1.2fr_1.5fr_0.6fr]', 'grow', 'h-0', 'h-0.

In [14]:
scripts = soup.find_all("script")

print("Total script tags:", len(scripts))

for i, s in enumerate(scripts[:5]):
    print(f"\n--- SCRIPT {i} ---")
    print(s.text[:1000])


Total script tags: 50

--- SCRIPT 0 ---
{"@context":"https://schema.org","@type":"Organization","image":"https://static.cricbuzz.com/images/cb_logo.svg","url":"https://www.cricbuzz.com","sameAs":["https://m.cricbuzz.com/","https://www.youtube.com/@cricbuzz","https://x.com/cricbuzz","https://www.facebook.com/cricbuzz/","https://www.instagram.com/cricbuzzofficial/"],"logo":"https://static.cricbuzz.com/images/cb_logo.svg","name":"Cricbuzz.com (Cricbuzz Platforms LIMITED)","description":"Cricbuzz.com works to provide you with blazing fast, reliable and accurate cricket updates of all live cricket news, international cricket matches as well as the cricket leagues from around the world (Indian Premier League, Big Bash League, Champions League T20 .. etc). Our team of cricket commentators, editors and technicians ensure you get the best possible updates about a live cricket match or breaking cricket news in a snap.","email":"support@cricbuzz.com","telephone":"+91-080-26790623","address":[{"@t

In [26]:
# ==========================================================
#  PLAYER DESCRIPTIVE METADATA SCRAPER (CRICBUZZ, LOCAL)
#  JSON-LD BASED | JUPYTER READY | RESTART SAFE
# ==========================================================

import pandas as pd
import requests
import json
import re
import time
import os
from tqdm import tqdm

# ----------------------------------------------------------
# 1. PATH CONFIGURATION (EDIT THESE IF NEEDED)
# ----------------------------------------------------------

DELIVERIES_PATH = r"C:\Users\HPP\Documents\Final Year Project\BowlerIQ\cricsheet_project_cricsheet_data\data\processed\deliveries.csv"
NAMES_PATH = r"C:\Users\HPP\Documents\Final Year Project\BowlerIQ\names (1).csv"
PEOPLE_PATH = r"C:\Users\HPP\Documents\Final Year Project\BowlerIQ\people (1).csv"

OUTPUT_FILE = r"C:\Users\HPP\Documents\Final Year Project\BowlerIQ\player_profile_raw.csv"

import unicodedata
import re

def normalize_name(name):
    if pd.isna(name):
        return None
    name = unicodedata.normalize("NFKD", str(name))
    name = name.encode("ascii", "ignore").decode("ascii")
    name = name.lower()
    name = re.sub(r"\.", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

# Normalize deliveries bowler names
deliveries["bowler_norm"] = deliveries["bowler"].apply(normalize_name)
unique_bowlers = set(deliveries["bowler_norm"].dropna())

# Normalize registry names
names["name_norm"] = names["name"].apply(normalize_name)

# Match using normalized aliases
player_names = names[names["name_norm"].isin(unique_bowlers)]

# Merge with people registry
player_map = player_names.merge(
    people[["identifier", "unique_name"]],
    on="identifier",
    how="left"
)[["identifier", "unique_name"]].drop_duplicates()

player_map.rename(columns={
    "identifier": "cricsheet_player_id",
    "unique_name": "player_name"
}, inplace=True)

print("Players resolved via registry:", len(player_map))

# ----------------------------------------------------------
# 3. RESUME SUPPORT
# ----------------------------------------------------------

if os.path.exists(OUTPUT_FILE):
    scraped = pd.read_csv(OUTPUT_FILE)
    done_ids = set(scraped["cricsheet_player_id"])
    print("Resuming — already scraped:", len(done_ids))
else:
    done_ids = set()

# ----------------------------------------------------------
# 4. HELPER FUNCTIONS
# ----------------------------------------------------------

HEADERS = {"User-Agent": "Mozilla/5.0"}

def build_cricbuzz_url(name):
    slug = name.lower().replace(" ", "-")
    return f"https://www.cricbuzz.com/profiles/{slug}"

def extract_player_json(html):
    blocks = re.findall(
        r'<script type="application/ld\+json">(.*?)</script>',
        html, re.S
    )
    for block in blocks:
        try:
            data = json.loads(block)
            if isinstance(data, dict) and "mainEntity" in data:
                entity = data["mainEntity"]
                if entity.get("type") == "Person":
                    return entity
        except:
            continue
    return None

def infer_bowling_arm(text):
    if not text:
        return None
    t = text.lower()
    if "left-arm" in t:
        return "left"
    if "right-arm" in t:
        return "right"
    return None

# ----------------------------------------------------------
# 5. SCRAPING LOOP
# ----------------------------------------------------------

rows = []

for _, r in tqdm(player_map.iterrows(), total=len(player_map)):
    pid = r["cricsheet_player_id"]
    name = r["player_name"]

    if pid in done_ids:
        continue

    url = build_cricbuzz_url(name)

    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        if resp.status_code != 200:
            raise Exception(f"HTTP {resp.status_code}")

        person = extract_player_json(resp.text)
        if not person:
            raise Exception("No JSON-LD person found")

        description = person.get("description", "")

        rows.append({
            "cricsheet_player_id": pid,
            "player_name": person.get("name"),
            "cricbuzz_profile_url": url,
            "date_of_birth": person.get("birthDate"),
            "birth_place": person.get("birthPlace"),
            "nationality": person.get("nationality"),
            "role": person.get("jobTitle"),
            "bowling_arm": infer_bowling_arm(description),
            "source": "cricbuzz"
        })

    except Exception as e:
        rows.append({
            "cricsheet_player_id": pid,
            "player_name": name,
            "cricbuzz_profile_url": url,
            "error": str(e),
            "source": "cricbuzz"
        })

    # Write incrementally
    if len(rows) >= 20:
        pd.DataFrame(rows).to_csv(
            OUTPUT_FILE,
            mode="a",
            header=not os.path.exists(OUTPUT_FILE),
            index=False
        )
        rows = []

    time.sleep(1.2)  # polite scraping

# Save leftovers
if rows:
    pd.DataFrame(rows).to_csv(
        OUTPUT_FILE,
        mode="a",
        header=not os.path.exists(OUTPUT_FILE),
        index=False
    )

print("=======================================")
print("SCRAPING COMPLETE")
print("Saved to:", OUTPUT_FILE)
print("=======================================")


Players resolved via registry: 52
Resuming — already scraped: 51


100%|██████████████████████████████████████████████████████████████████████████████████| 52/52 [00:02<00:00, 20.35it/s]

SCRAPING COMPLETE
Saved to: C:\Users\HPP\Documents\Final Year Project\BowlerIQ\player_profile_raw.csv


In [1]:
import sys
print(sys.executable)


C:\Users\HPP\anaconda3\envs\mri_ml\python.exe


In [2]:
import sys
print(sys.version)


3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]


In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time

# -------------------------
# Chrome Options
# -------------------------
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

# Optional (run headless if you want)
# options.add_argument("--headless")

# -------------------------
# Launch Browser (AUTO driver)
# -------------------------
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# -------------------------
# Test search
# -------------------------
player_name = "Jasprit Bumrah"
query = f'site:cricbuzz.com/profiles "{player_name}"'
url = f"https://duckduckgo.com/?q={query}"

print("Opening:", url)
driver.get(url)

time.sleep(3)

links = driver.find_elements(By.TAG_NAME, "a")

profile_urls = []
for link in links:
    href = link.get_attribute("href")
    if href and "cricbuzz.com/profiles/" in href:
        profile_urls.append(href)

print("\nFound profiles:")
for p in set(profile_urls):
    print(p)

# driver.quit()  # Uncomment when done


Opening: https://duckduckgo.com/?q=site:cricbuzz.com/profiles "Jasprit Bumrah"

Found profiles:
https://www.cricbuzz.com/profiles/8117/trent-boult
https://www.cricbuzz.com/profiles/13217/arshdeep-singh
https://www.cricbuzz.com/profiles/8117/trent-boult#!#profile
https://www.cricbuzz.com/profiles/7836/deepak-chahar
https://www.cricbuzz.com/profiles/9311/jasprit-bumrah/all-matches/batting
https://www.cricbuzz.com/profiles/9311/jasprit-bumrah


In [6]:
import re
from difflib import SequenceMatcher

def normalize_name(name):
    return name.lower().replace(" ", "-")

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def select_best_profile(player_name, urls):
    target = normalize_name(player_name)

    best_match = None
    best_score = 0

    for url in urls:
        match = re.search(r"/profiles/\d+/([^/?]+)", url)
        if not match:
            continue

        slug = match.group(1)
        score = similarity(slug, target)

        if score > best_score:
            best_score = score
            best_match = url

    return best_match, best_score


In [7]:
player = "Jasprit Bumrah"
urls = [
    "https://www.cricbuzz.com/profiles/8117/trent-boult",
    "https://www.cricbuzz.com/profiles/13217/arshdeep-singh",
    "https://www.cricbuzz.com/profiles/9311/jasprit-bumrah"
]

best_url, score = select_best_profile(player, urls)

print("Selected:", best_url)
print("Confidence:", round(score, 2))


Selected: https://www.cricbuzz.com/profiles/9311/jasprit-bumrah
Confidence: 1.0


In [11]:
# ===========================
# CRICBUZZ PROFILE MAPPER
# ===========================

import pandas as pd
import time
import re
from difflib import SequenceMatcher

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


# ---------------------------
# CONFIG
# ---------------------------
INPUT_CSV = r"C:\Users\HPP\Documents\Final Year Project\BowlerIQ\player_metadata_raw.csv"
OUTPUT_CSV = "players_with_cricbuzz_profile.csv"
SEARCH_DELAY = 3
MATCH_THRESHOLD = 0.55

# ---------------------------
# UTILITIES
# ---------------------------

def extract_last_name(name):
    """
    Extracts the last meaningful token.
    Handles cases like:
    - 'V Kohli' -> Kohli
    - 'AA Barot' -> Barot
    - 'M J Clarke' -> Clarke
    """
    parts = name.strip().split()
    meaningful = [p for p in parts if not (len(p) <= 2 and p.isupper())]
    return meaningful[-1] if meaningful else parts[-1]


def normalize(text):
    return re.sub(r"[^a-z]", "", text.lower())


def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()


def best_profile_match(player_name, urls):
    best_url = None
    best_score = 0

    clean_player = normalize(player_name)

    for url in urls:
        match = re.search(r"/profiles/\d+/([^/?]+)", url)
        if not match:
            continue

        slug = match.group(1)
        slug_clean = normalize(slug)

        score = similarity(clean_player, slug_clean)

        if score > best_score:
            best_score = score
            best_url = url

    return best_url, round(best_score, 3)


# ---------------------------
# SELENIUM SETUP
# ---------------------------
options = Options()
options.add_argument("--headless")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# ---------------------------
# LOAD DATA
# ---------------------------
df = pd.read_csv(INPUT_CSV)
results = []

# ---------------------------
# MAIN LOOP
# ---------------------------
for _, row in df.iterrows():
    player = row["player_name"]
    last_name = extract_last_name(player)

    print(f"\n🔍 Searching for: {player}")

    query = f'site:cricbuzz.com/profiles "{last_name}"'
    search_url = f"https://duckduckgo.com/?q={query}"

    driver.get(search_url)
    time.sleep(3)

    links = driver.find_elements(By.TAG_NAME, "a")
    candidates = []

    for link in links:
        href = link.get_attribute("href")
        if href and "cricbuzz.com/profiles/" in href:
            candidates.append(href)

    best_url, score = best_profile_match(player, candidates)

    results.append({
        "player_name": player,
        "matched_profile": best_url,
        "confidence_score": score
    })

    print(f"   → Match: {best_url} | Score: {score}")

    time.sleep(SEARCH_DELAY)

# ---------------------------
# SAVE OUTPUT
# ---------------------------
df_out = pd.DataFrame(results)
df_out.to_csv(OUTPUT_CSV, index=False)

driver.quit()

print("\n✅ DONE — Output saved to:", OUTPUT_CSV)



🔍 Searching for: AAA Amsterdam
   → Match: None | Score: 0

🔍 Searching for: AA Adeoye
   → Match: None | Score: 0

🔍 Searching for: AA Alleyne
   → Match: None | Score: 0

🔍 Searching for: AAA Patel
   → Match: None | Score: 0

🔍 Searching for: AAA White
   → Match: None | Score: 0

🔍 Searching for: AA Baig
   → Match: None | Score: 0

🔍 Searching for: AA Bamal
   → Match: None | Score: 0

🔍 Searching for: AA Banner
   → Match: None | Score: 0

🔍 Searching for: AA Barot
   → Match: None | Score: 0


KeyboardInterrupt: 

In [13]:
import requests
import json
import re
from difflib import SequenceMatcher

def search_commoncrawl(query):
    url = (
        "https://index.commoncrawl.org/CC-MAIN-2023-50-index"
        f"?url=cricbuzz.com/profiles*&output=json"
    )

    response = requests.get(url, timeout=30)
    if response.status_code != 200:
        return []

    results = []
    for line in response.text.splitlines():
        data = json.loads(line)
        url = data.get("url", "")
        if query.lower() in url.lower():
            results.append(url)

    return list(set(results))


In [14]:
def find_best_profile(player_name):
    tokens = player_name.lower().split()
    results = search_commoncrawl(" ".join(tokens))

    best_score = 0
    best_url = None

    for url in results:
        slug = url.split("/profiles/")[-1]
        score = SequenceMatcher(None, slug, "-".join(tokens)).ratio()
        if score > best_score:
            best_score = score
            best_url = url

    return best_url, best_score


In [15]:
player = "V Kohli"
profile, score = find_best_profile(player)

print("Profile:", profile)
print("Confidence:", score)


Profile: None
Confidence: 0


In [17]:
import pandas as pd

# Load the deliveries data
df = pd.read_csv(
    r"C:\Users\HPP\Documents\Final Year Project\BowlerIQ\cricsheet_project_cricsheet_data\data\processed\deliveries.csv"
)

# Extract unique bowler names
unique_bowlers = (
    df["bowler"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

# Convert to DataFrame
bowler_df = pd.DataFrame({"player_name": sorted(unique_bowlers)})

# Save to CSV
output_path = r"C:\Users\HPP\Documents\Final Year Project\BowlerIQ\bowler_list_unique.csv"
bowler_df.to_csv(output_path, index=False)

print(f"Total unique bowlers found: {len(bowler_df)}")
print("Saved to:", output_path)


Total unique bowlers found: 3525
Saved to: C:\Users\HPP\Documents\Final Year Project\BowlerIQ\bowler_list_unique.csv


In [19]:
bowler_df.head()
bowler_df.tail()

,player_name
3520,zuhaib zubair
3521,zulfiqar babar
3522,zulqarnain haider
3523,zulqarnain haider (2)
3524,zxm vukusic


In [25]:
import requests
import json

player_id = 35320  # example

url = f"https://site.web.api.espn.com/apis/common/v3/sports/cricket/athletes/{player_id}"
resp = requests.get(url)

data = resp.json()

athlete = data.get("athlete", {})

print("Name:", athlete.get("displayName"))
print("Country:", athlete.get("country", {}).get("name"))
print("Role:", athlete.get("position", {}).get("name"))
print("Batting Style:", athlete.get("battingStyle"))
print("Bowling Style:", athlete.get("bowlingStyle"))


Name: Sachin Ramesh Tendulkar
Country: None
Role: Top-order batter
Batting Style: None
Bowling Style: None


In [26]:
import requests
import json

player_id = 35320  # example
url = f"https://site.web.api.espn.com/apis/common/v3/sports/cricket/athletes/{player_id}"

resp = requests.get(url)
data = resp.json()

print(data.keys())


dict_keys(['athlete', 'playerSwitcher', 'links', 'ticketsInfo', 'videos'])


In [31]:
import requests
import json

# Replace with any valid ESPN player ID
player_id = 35320   # Example: Sachin Tendulkar

url = f"https://site.web.api.espn.com/apis/common/v3/sports/cricket/athletes/{player_id}"

response = requests.get(url)
data = response.json()

# Print full raw JSON (for inspection)
print(json.dumps(data, indent=2))

# ---- Extract useful fields ----
athlete = data.get("athlete", {})

print("\n--- Parsed Fields ---")
"player_id": a.get("id"),
"name": a.get("displayName"),
        "country": a.get("team", {}).get("displayName"),
        "role": a.get("position", {}).get("name"),
        "batting_style": a.get("battingStyle"),
        "bowling_style": a.get("bowlingStyle")


{
  "athlete": {
    "id": "35320",
    "uid": "s:200~a:35320",
    "guid": "a1acb553-2846-5228-8118-b17630166ee8",
    "type": "cricket",
    "firstName": "Sachin",
    "lastName": "Tendulkar",
    "displayName": "Sachin Ramesh Tendulkar",
    "fullName": "Sachin Ramesh Tendulkar",
    "shortName": "Sachin Tendulkar",
    "links": [
      {
        "language": "en",
        "rel": [
          "playercard",
          "desktop"
        ],
        "href": "https://www.espncricinfo.com/ci/content/player/35320.html",
        "text": "Player Card",
        "shortText": "Player Card",
        "isExternal": false,
        "isPremium": false
      }
    ],
    "headshot": {
      "href": "https://a.espncdn.com/i/headshots/cricket/players/full/35320.png",
      "rel": [
        "full",
        "default"
      ]
    },
    "position": {
      "id": "TBT",
      "name": "Top-order batter",
      "abbreviation": "TBT",
      "leaf": true,
      "parent": {
        "leaf": false
      },
      "slu

In [3]:
import pandas as pd
df = pd.read_csv(
    r"C:\Users\HPP\Documents\Final Year Project\BowlerIQ\cricsheet_project_cricsheet_data\data\processed\deliveries.csv"
)

In [4]:
df.head()

,match_id,delivery_id,innings,over,ball_in_over,bowler,batsman,non_striker,runs_off_bat,extras,wicket,wicket_kind,wicket_player,phase
0,1203674,1203674_1_0_1,1,0,1,ss cottrell,pr stirling,gj delany,0,0,0,NaN,NaN,powerplay
1,1203674,1203674_1_0_2,1,0,2,ss cottrell,pr stirling,gj delany,4,0,0,NaN,NaN,powerplay
2,1203674,1203674_1_0_3,1,0,3,ss cottrell,pr stirling,gj delany,1,0,0,NaN,NaN,powerplay
3,1203674,1203674_1_0_4,1,0,4,ss cottrell,gj delany,pr stirling,0,0,0,NaN,NaN,powerplay
4,1203674,1203674_1_0_5,1,0,5,ss cottrell,gj delany,pr stirling,2,0,0,NaN,NaN,powerplay
